# 这是一个VASP学习笔记

## **VASP核心原理**

<img src="./VASP原理.png" width="40%">

大概流程：  
step 1：先优化电子分布(粒子实$R_i$的分布预设定且保持不变)。初始化电荷密度$n_0(r)$，计算出kohn-sham势$v_{KS}(r)$，代入kohn-sham方程，解出波函数进而求出电荷密度$n(r)$，和前述电荷密度对比，观察是否收敛；若否，则继续迭代，直至收敛，自洽循环完成。  
step 2：再优化离子实分布。利用step 1算出的电荷密度计算各离子实的受力，并进行一步优化；  
step 3：因为step 2中离子实位置改变，因此又重新回到step 1计算平衡电荷密度分布，依次交替优化电子电荷以及离子实分布，直至离子实的受力为零，此时结构优化结束。

## **VASP常用文件**

### 1. 输入文件

**INCAR**：计算控制参数文件  
* ISTART：设置初始波函数<br>
0：随机生成波函数<br>
1：从WAVECAR读取波函数，若读取失败则随机生成<br>
* ICHARG：设置初始电荷密度<br>
0：通过 WAVECAR 生成电荷密度，若读取失败则通过原子电荷密度叠加生成<br>
1：从CHGCAR读取电荷密度<br>
2：通过原子电荷密度叠加生成电荷密度<br>
11：从CHGCAR读取电荷密度，且在自洽循环中保持不变<br>
* ENCUT：平面波截断能(单位eV)，决定了近似展开所用平面波的最大能量上限(也即波矢上限)<br>
一般为POTCAR中ENMAX参数值的1.0~1.3倍<br>
* PREC：总体计算精度<br>
可设为 Low、Med、High、Normal、Single、Accurate(注：PREC中涉及到ENCUT的设置，通过重新设置ENCUT可覆盖该设置，而且通常ENCUT采用自定义设置)<br>
<img src="./PREC.png" width="40%"><br>
* ISPIN：自旋极化<br>
1：非自旋极化(非磁性体系，每个轨道自旋向上与向下的电子数相等)<br>
2：自旋极化(适合铁磁、反铁磁体系)<br>
* MAGMOM：原子初始磁矩(需要与POSCAR的原子顺序对应，合理选择可设置成铁磁、反铁磁等体系)<br>
每个原子默认值为1，对于含有d、f电子的原子可设置较大的值(d和f轨道最大磁矩为5和7)<br>
* NCORE：并行计算单个轨道所用的核心数量<br>
* ALGO：电子波函数优化算法<br>
N：DAV算法，收敛性好，速度慢<br>
V：RMM算法，收敛性差，速度快(大概比N快2~3倍)<br>
F：上述两种算法结合，表现与V类似(用得较少)<br>
* EDIFF：自洽循环能量收敛标准，单位为eV<br>
一般设置为1E-5~1E-6<br>
* NELM：自洽循环最大计算步数
实际计算步数，自旋极化体系大于非极化体系；维数越低，计算步数越高(三维体系一般设置为50~100即可，二维体系可设为200)<br>
* AMIX：自洽循环中电荷密度混合时新电荷密度的含量<br>
收敛不好时可降低数值(让新电荷少掺入)<br>
* IBRION：离子位置优化算法<br>
0：分子动力学模拟<br>
1：准牛顿法<br>
2：共轭梯度法<br>
5：振动频率计算<br>
6：弹性常数计算<br>
(结构优化中常用1和2，其中1速度快，2收敛性更好)<br>
* EDIFFG：离子位置优化收敛标准<br>
正值为系统能量变化，负值为原子上残余力；随着体系维数下降，可适当增加原子残余力<br>
* NSW：离子位置优化最大步数(IBRION=1、2)、分子动力学模拟步数(IBRION=0)<br>
* ISIF：结构优化内容<br>
<img src="./ISIF.png" width="40%"><br>
一般三维结构优化设成3，算表面或缺陷时设成2，二维材料设成2<br>
注意：ISIF=2结构优化时不会改变晶格常数，如果已经确定了晶格常数，就用2；否则，想要优化晶格常数，就用3</font><br>
* ISMEAR：轨道分数占据<font color=red>???</font><br>
-5：四面体方法(适合半导体和绝缘体)<br>
0：高斯方法(适合导体、半导体和绝缘体)，展宽由SIGMA确定<br>
N：Methfessel-Paxton方法(适合导体)
* NEDOS：态密度数据点个数<br>
一般取1000~3000<br>
* LORBIT：控制了VASP计算中关于电子波函数投影的详细输出，各参数详见<https://www.vasp.at/wiki/index.php/LORBIT>(设为11时，将在OUTCAR中输出局域磁矩)<br>
* LSORBIT：控制是否开启<font color=red>自旋-轨道耦合(SOC)</font>，默认为False；当开启时，需要使用vasp_ncl来编译<br>

**POSCAR**：描述体系结构的参数<br>
<img src="./POSCAR_1.png" width="70%"> <img src="POSCAR_2.png" width="70%"><br>

**KPOINTS**：k点取样设置文件<br>
* 自动产生k点：方式2比较常用，其中倒空间三个常数取值$N_1$、$N_2$、$N_3$应当取决于实空间晶格常数a、b、c的数值，最好保持它们对应的乘积相等(相当于使得倒空间各k点可以将晶胞划分为许多小正方体)，且数值在20~30范围；另外，$N_1$、$N_2$、$N_3$最好取奇数，以包含$\Gamma$点；金属相比半导体和绝缘体需要取更多的k点<br>
<img src="./KPOINTS_1.png" width="50%">
* 手动输入各高对称k点：多用于计算特定能带；R开头表示以倒格子为基矢的坐标值，C开头表示以原正交坐标系$\vec{e_1}$、$\vec{e_2}$、$\vec{e_3}$为基矢的坐标值，即：$\vec{k}=\frac{2\pi}{a} (x_1,x_2,x_3)$，坐标为$(x_1,x_2,x_3)，$其中$a$为**POSCAR**第二行设置的缩放因子<br>
<img src="./KPOINTS_2.png" width="50%"><br>

**POTCAR**：赝势文件<br>
通常各元素的POTCAR文件已经包含在VASP软件包中的赝势库里了，只要确定自己需要哪些原子的赝势，按照POSCAR的顺序连接合成一个文件即可<br>
<img src="./POTCAR.png" width="50%"><br>

### 2. 输出文件
**OUTCAR**：最主要的输出文件<br>
**DOSCAR**：电子态密度文件<br>
**WAVECAR**：波函数文件<br>
**CHGCAR**：电荷密度文件<br>
**EIGENVAL**：各k点本征值文件，能带数量，能带范围

## **VASP计算流程**

### 1. 准备POSCAR
需要研究某特定结构的晶体，可先去Material Project网站找到该晶体，下载其cif文件。之后用VESTA打开该cif文件，可查看其晶胞结构。之后可用该软件将其以vasp的POSCAR格式导出，可选择导出原胞的数据或者其他类型(ps:想把晶胞数据转化成原胞，还可通过vaspkit实现)

### 2. 准备INCAR、KPOINTS、POTCAR
INCAR等文件可以使用vaspkit基于POSCAR自动导出；之后再修改INCAR文件的内容；POTCAR有时可以自己选择生成，可优化运行速度

### 3. 晶格结构优化
由于是从其他地方下载的POSCAR数据，并不一定准确，因此需要先进行晶格优化，使原胞各离子的位置达到最优。该步骤无需产生WAVECAR和CHGCAR，所以INCAR中的相应设置均为"False"，同时NSW要设置数据。输入`mpirun -np 6 vasp_std`开始运行，结束后产生优化结果<br>
结构优化时间以及效果的影响因素：EDIFF、EDIFFG、ENCUT、ISIF、KPOINTS点密度、POTIM；可采用多步优化法<br>
困难点：若参数设置不合理(精度过高)，移动步长(POTIM)不合理，或者初始POSCAR距离标准解太远，达到收敛会很耗时间<br>
经验参考：<https://blog.sciencenet.cn/blog-567091-895440.html>

### 4. 自洽计算
该步骤的目的是根据离子实分布计算出最合理的波函数以及电荷密度分布。做法是将上一步产生的CONTCAR文件复制到自洽计算文件夹中，重命名为POSCAR；把前面的KPOINTS、POTCAR、INCAR文件也完全复制过来，其中只需要修改INCAR文件：把NSW设置为0，LWAVE、LCHARG设置为"TRUE"。输入`mpirun -np 6 vasp_std`开始运行，结束后产生计算结果

### 5. 非自洽计算
该步骤用于各种性质的计算，如能带等。做法是将自洽计算产生的WAVECAR、CHGCAR复制到当前文件夹下，POSCAR、POTCAR都与之前相同，只有KPOINTS、INCAR需要修改。INCAR的LWAVE、LCHARG设置为"False"，ICHARG设置为11(读取电荷密度)，LORBIT设置为11。计算能带需要得到高对称点，可以通过vaspkit实现(选择3:K-Path for Band-Structure)，之后依据导出的高对称点文件KPATH.in来写KPOINTS文件即可。输入`mpirun -np 6 vasp_std`开始运行，结束后产生优化结果。之后可用vaspkit直接产生能带图(选择21: Band-Structure)

## **VASP磁性计算**
根据VASP的说明书，跟自旋极化有关的主要是ISPIN、LORBIT、MAGMOM。其中ISPIN=2 是考虑自旋极化，LORBIT=11将在OUTCAR中输出局域磁矩，MAGMOM是设置初始磁矩，比如铁磁和反铁磁。设置MAGMOM时，可以读入无磁的WAVECAR和CHGCAR，所以计算的时候通常是鼓励先优化，不考虑自旋极化来优化，把原子优化好了以后，再考虑自旋极化设置不同的磁构型。